In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pickle

from src.data_utils import (
    load_csv_plan,
    load_tabular_features,
    make_class_mappings,
    load_logmel_features,
)
from sklearn.ensemble import RandomForestClassifier

from src.models import svm_model, random_forest_model
from src.evaluation import run_predefined_fold_cv, summarize_fold_metrics
from src.cnn import train_cnn_cv

import torch
import optuna

ROOT = Path.cwd().parent.resolve()
DATA_DIR = ROOT / "data"

FEATURE_PATH = DATA_DIR / "features/tabular_audio_features.npz"
LOGMEL_PATH = DATA_DIR / "features/logmel_spectogram.npz"
CV_PLAN_PATH = DATA_DIR / "processed/cv_plan_10fold.json"
OUTPUT_DIR = ROOT / "results/models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Loading Processed Data

In [ ]:
data = load_tabular_features(FEATURE_PATH)
cv_plan = load_csv_plan(CV_PLAN_PATH)

X = data["X"].astype(np.float32)
y = data["y"].astype(np.int64)
labels = data["labels"]
folds = data["folds"].astype(np.int64)
filenames = data["filenames"]
feature_names = data["feature_names"]

print("X:", X.shape)
print("Y:", y.shape)
print("folds:", folds.shape)
print("CV folds:", len(cv_plan))

class_table, class_ids, class_names, id_to_class = make_class_mappings(y, labels)
display(class_table)

num_classes = len(class_ids)

In [ ]:
data_logmel = load_logmel_features(LOGMEL_PATH)

X_logmel = data_logmel["X"].astype(np.float32)
y_logmel = data_logmel["y"].astype(np.int64)
labels_logmel = data_logmel["labels"]
folds_logmel = data_logmel["folds"].astype(np.int64)
filenames_logmel = data_logmel["filenames"]

print("X:", X_logmel.shape)
print("Y:", y_logmel.shape)
print("folds:", folds_logmel.shape)

## Part 1: SVM Tuning

In [ ]:
svm_experiments = [
    {"name": "baseline", "C": 10, "gamma": "scale"},
    {"name": "low_C", "C": 1, "gamma": "scale"},
    {"name": "high_C", "C": 100, "gamma": "scale"},
    {"name": "gamma_001", "C": 10, "gamma": 0.01},
    {"name": "gamma_0001", "C": 10, "gamma": 0.001},
]

results = []

best_score = -1
best_fold_metrics = None
best_predictions = None
best_summary = None
best_experiment = None

for experiment in svm_experiments:
    print("Experiment Config:")
    print(experiment, end="\n\n")
    fold_metrics, predictions_df = run_predefined_fold_cv(
        model_fn=lambda: svm_model(
            seed=42, C=experiment["C"], gamma=experiment["gamma"]
        ),
        X=X,
        y=y,
        folds=folds,
        filenames=filenames,
        cv_plan=cv_plan,
        id_to_class=id_to_class,
    )

    summary = summarize_fold_metrics(fold_metrics)

    results.append({"experiment": experiment["name"], **summary})
    print("\n\n")

    if summary["f1_macro_mean"] > best_score:
        best_score = summary["f1_macro_mean"]
        best_fold_metrics = fold_metrics
        best_predictions = predictions_df
        best_summary = summary
        best_experiment = experiment

svm_results = pd.DataFrame(results)

In [ ]:
svm_output = OUTPUT_DIR / "svm"
svm_output.mkdir(parents=True, exist_ok=True)

svm_results.to_csv(svm_output / "tuning_results.csv", index=False)

best_fold_metrics.to_csv(svm_output / "fold_metrics.csv", index=False)
best_predictions.to_csv(svm_output / "predictions.csv", index=False)

with open(svm_output / "summary_metrics.json", "w") as f:
    json.dump(best_summary, f, indent=4)

with open(svm_output / "best_config.json", "w") as f:
    json.dump(best_experiment, f, indent=4)

## Part 2: Random Forest

In [ ]:
rf_experiments = [
    {"name": "baseline", "n_estimators": 300, "max_depth": None, "min_sample_leaf": 1},
    {
        "name": "more_trees",
        "n_estimators": 500,
        "max_depth": None,
        "min_sample_leaf": 1,
    },
    {
        "name": "limited_depth",
        "n_estimators": 300,
        "max_depth": 30,
        "min_sample_leaf": 1,
    },
    {
        "name": "regularized_leaf2",
        "n_estimators": 300,
        "max_depth": None,
        "min_sample_leaf": 2,
    },
    {
        "name": "regularized_leaf4",
        "n_estimators": 300,
        "max_depth": None,
        "min_sample_leaf": 4,
    },
]

results = []

best_score = -1
best_fold_metrics = None
best_predictions = None
best_summary = None
best_experiment = None

for experiment in rf_experiments:
    print("Experiment Config:")
    print(experiment, end="\n\n")
    fold_metrics, predictions_df = run_predefined_fold_cv(
        model_fn=lambda: random_forest_model(
            seed=42,
            n_estimators=experiment["n_estimators"],
            max_depth=experiment["max_depth"],
            min_samples_leaf=experiment["min_sample_leaf"],
        ),
        X=X,
        y=y,
        folds=folds,
        filenames=filenames,
        cv_plan=cv_plan,
        id_to_class=id_to_class,
    )

    summary = summarize_fold_metrics(fold_metrics)

    results.append({"experiment": experiment["name"], **summary})
    print("\n\n")

    if summary["f1_macro_mean"] > best_score:
        best_score = summary["f1_macro_mean"]
        best_fold_metrics = fold_metrics
        best_predictions = predictions_df
        best_summary = summary
        best_experiment = experiment

rf_results = pd.DataFrame(results)

In [ ]:
rf_final = random_forest_model(
    seed=42,
    n_estimators=best_experiment["n_estimators"],
    max_depth=best_experiment["max_depth"],
    min_samples_leaf=best_experiment["min_sample_leaf"],
)
rf_final.fit(X, y)

In [ ]:
importance = rf_final.feature_importances_

importance_df = pd.DataFrame(
    {"feature": feature_names, "importance": importance}
).sort_values("importance", ascending=False)

In [ ]:
rf_output = OUTPUT_DIR / "random_forest"
rf_output.mkdir(parents=True, exist_ok=True)

rf_results.to_csv(rf_output / "tuning_results.csv", index=False)
importance_df.to_csv(rf_output / "rf_feature_importance.csv", index=False)

best_fold_metrics.to_csv(rf_output / "fold_metrics.csv", index=False)
best_predictions.to_csv(rf_output / "predictions.csv", index=False)

with open(rf_output / "summary_metrics.json", "w") as f:
    json.dump(best_summary, f, indent=4)

with open(rf_output / "best_config.json", "w") as f:
    json.dump(best_experiment, f, indent=4)

## Part 3: CNN

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


def objective(trial):
    config = {
        "seed": 42,
        "batch_size": trial.suggest_categorical("batch_size", [64, 128]),
        "max_epoch": 12,
        "patience": 3,
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True),
        "dropout": trial.suggest_float("dropout", 0.2, 0.6),
        "validation_size": 0.2,
        "num_workers": 2,
    }

    fold_metrics_df, _, _, _, _ = train_cnn_cv(
        X_logmel,
        y_logmel,
        folds_logmel,
        filenames_logmel,
        cv_plan,
        id_to_class,
        num_classes,
        device,
        config,
        verbose=False,
        test_folds=[1, 5, 9],
    )

    return fold_metrics_df["f1_macro"].mean()

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

print("Best value:", study.best_value)
print("Best params:", study.best_params)

In [ ]:
best_cnn_config = {
    "seed": 42,
    "max_epoch": 40,
    "patience": 7,
    "validation_size": 0.2,
    "num_workers": 2,
    **study.best_params,
}

In [ ]:
fold_metrics_df, predictions_df, histories, best_state, best_info = train_cnn_cv(
    X=X_logmel,
    y=y_logmel,
    folds=folds_logmel,
    filenames=filenames_logmel,
    cv_plan=cv_plan,
    id_to_class=id_to_class,
    num_classes=num_classes,
    device=device,
    config=best_cnn_config,
    verbose=True,
)

In [ ]:
summary_metrics = summarize_fold_metrics(fold_metrics_df)

out_path = OUTPUT_DIR / "cnn"
out_path.mkdir(parents=True, exist_ok=True)

fold_metrics_df.to_csv(out_path / "fold_metrics.csv", index=False)
predictions_df.to_csv(out_path / "predictions.csv", index=False)

with open(out_path / "summary_metrics.json", "w") as f:
    json.dump(summary_metrics, f, indent=4)

with open(out_path / "best_config.json", "w") as f:
    json.dump(best_cnn_config, f, indent=4)

with open(out_path / "histories.pk1", "wb") as f:
    pickle.dump(histories, f)

torch.save(best_state, out_path / "best_model.pt")

optuna_results = study.trials_dataframe()

optuna_results.to_csv(out_path / "optuna_results.csv", index=False)